## Chat Models - <a href='https://python.langchain.com/docs/modules/data_connection/document_loaders/'>Document Loaders</a> and Text Splitting


In [ ]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters langsmith beautifulsoup4 langchain-classic --upgrade

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set USER_AGENT for web requests
os.environ["USER_AGENT"] = "langchain-doc-loaders-notebook"

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [11]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import TextLoader
import requests

# Get this file and save it locally:x
url = "https://github.com/hammer-mt/thumb/blob/master/README.md"

# Save it locally:
r = requests.get(url)

# Extract the text from the HTML:
soup = BeautifulSoup(r.text, 'html.parser')
text = soup.get_text()

with open("README.md", "w") as f:
    f.write(text)

loader = TextLoader('README.md')
docs = loader.load()

In [12]:
from langchain_core.documents import Document
[ Document(page_content='test', metadata={'test': 'test'}) ] 

[Document(metadata={'test': 'test'}, page_content='test')]

In [13]:
# Split the text into sentences:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size = 300,
    chunk_overlap  = 50,
    length_function = len,
    is_separator_regex = False,
)

final_docs = text_splitter.split_documents(loader.load())

In [14]:
len(final_docs)

4

In [15]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.chains.summarize import load_summarize_chain

chat = ChatOpenAI()

In [16]:
chain = load_summarize_chain(llm=chat, chain_type="map_reduce")
chain.invoke({
    "input_documents": final_docs,
})

{'input_documents': [Document(metadata={'source': 'README.md'}, page_content='Too many requests · GitHub'),
  Document(metadata={'source': 'README.md'}, page_content='Too many requests\nYou have exceeded a secondary rate limit.\n        Please wait a few minutes before you try again;\n        in some cases this may take up to an hour.\nSigning in may provide a higher rate limit if you are not already signed in.'),
  Document(metadata={'source': 'README.md'}, page_content='For more on scraping GitHub and how it may affect your rights, please review our Terms of Service.'),
  Document(metadata={'source': 'README.md'}, page_content='Contact Support —\n        GitHub Status —\n        @githubstatus')],
 'output_text': 'The error message "Too many requests · GitHub" suggests that the server is overloaded with requests and cannot fulfill current requests. Users should wait before trying again, sign in for a higher rate limit, refer to the Terms of Service for scraping guidelines, and follow 